pip install requests wikitextparser re os json

---
# Data Extraction from Terraria Wiki

In [1]:
import requests
import wikitextparser as wtp
import re
import os
import json
from kb_extraction import get_clean_terraria_wiki
from json_extract import save_to_manifest

## Guide:Getting Started

In [2]:
page_title = "Guide:Getting_started"
content = get_clean_terraria_wiki(page_title)
source_url = "https://terraria.wiki.gg/wiki/Guide:Getting_started"
game_state = "Pre-Hardmode"

In [3]:
with open("getting_started.txt", "w", encoding="utf-8") as file:
    file.write(content)

In [4]:
save_to_manifest(page_title, content, source_url, game_state)

Successfully saved manifest to: knowledge_base/processed/guide_getting_started.json


## Crafting 101

In [5]:
page_title = "Guide:Crafting_101"
content = get_clean_terraria_wiki(page_title)
source_url = "https://terraria.wiki.gg/wiki/Guide:Crafting_101"
game_state = "Pre-Hardmode"

In [6]:
with open("crafting101.txt", "w") as file:
    file.write(content)

In [7]:
save_to_manifest(page_title, content, source_url, game_state)

Successfully saved manifest to: knowledge_base/processed/guide_crafting_101.json


## Class Setups

In [8]:
page_title = "Guide:Class_setups"
content = get_clean_terraria_wiki(page_title)
source_url = "https://terraria.wiki.gg/wiki/Guide:Class_setups"
game_state = "Pre-Hardmode"

In [9]:
with open("class_setups.txt", "w") as file:
    file.write(content)

In [10]:
save_to_manifest(page_title, content, source_url, game_state)

Successfully saved manifest to: knowledge_base/processed/guide_class_setups.json


## Pre-Hardmode Walkthrough

In [11]:
page_title = "Guide:Walkthrough"
content = get_clean_terraria_wiki(page_title)
source_url = "https://terraria.wiki.gg/wiki/Guide:Walkthrough"
game_state = "Pre-Hardmode"

In [12]:
with open("Pre-Hardmode_Walkthrough.txt", "w") as file:
    file.write(content)

In [13]:
save_to_manifest(page_title, content, source_url, game_state)

Successfully saved manifest to: knowledge_base/processed/guide_walkthrough.json


## Getting started with hardmode

In [14]:
page_title = "Guide:Getting_started_with_Hardmode"
content = get_clean_terraria_wiki(page_title)
source_url = "https://terraria.wiki.gg/wiki/Guide:Getting_started_with_Hardmode"
game_state = "Hardmode"

In [15]:
with open("Getting_started_with_Hardmode.txt", "w") as file:
    file.write(content)

In [16]:
save_to_manifest(page_title, content, source_url, game_state)

Successfully saved manifest to: knowledge_base/processed/guide_getting_started_with_hardmode.json


## Hardmode walkthrough

In [17]:
page_title = "Guide:Walkthrough/Hardmode"
content = get_clean_terraria_wiki(page_title)
source_url = "https://terraria.wiki.gg/wiki/Guide:Walkthrough/Hardmode"
game_state = "Hardmode"

In [18]:
with open("Hardmode.txt", "w") as file:
    file.write(content)

In [19]:
save_to_manifest(page_title, content, source_url, game_state)

Successfully saved manifest to: knowledge_base/processed/guide_walkthrough_hardmode.json


## Practical Tips

In [20]:
page_title = "Guide:Practical_tips"
content = get_clean_terraria_wiki(page_title)
source_url = "https://terraria.wiki.gg/wiki/Guide:Practical_tips"
game_state = "Hardmode"

In [21]:
with open("Practical_tips.txt", "w") as file:
    file.write(content)

In [22]:
save_to_manifest(page_title, content, source_url, game_state)

Successfully saved manifest to: knowledge_base/processed/guide_practical_tips.json


## Bosses

In [24]:
page_title = "Bosses"
content = get_clean_terraria_wiki(page_title)
source_url = "https://terraria.wiki.gg/wiki/Bosses"
game_state = "Hardmode"

In [25]:
with open("Practical_tips.txt", "w") as file:
    file.write(content)

In [26]:
save_to_manifest(page_title, content, source_url, game_state)

Successfully saved manifest to: knowledge_base/processed/bosses.json


---
# Indexing

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, JSONLoader

folder_path = "./knowledge_base/processed"
loader = DirectoryLoader(
    path=folder_path, 
    glob="*.json",
    loader_cls=JSONLoader,
    loader_kwargs = {
        "jq_schema": ".content",
        "text_content": False
    }
)

docs = loader.load()

print(f"Loaded {len(docs)} files.")
print(f"Snippet of first file: {docs[0].page_content[:50]}...")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(docs)

print(f"Number of chunks created: {len(chunks)}")
print()

print("--- Chunk 0 (inspect before storing!) ---")
print(chunks[0].page_content)
print()
print("--- Chunk 1 ---")
print(chunks[1].page_content)

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

print("Embedding model loaded: all-MiniLM-L6-v2 (384-dimensional)")
print("Embedding all chunks into the vector database...")
db = FAISS.from_documents(chunks, embeddings)

print(f"Vector database created with {len(chunks)} vectors")

In [ ]:
db.save_local("faiss_index")
print("FAISS index saved to 'faiss_index/'")
print()
print("To reload later (skip the embedding step entirely):")
print("  db = FAISS.load_local('faiss_index', embeddings, allow_dangerous_deserialization=True)")

In [ ]:
retriever = db.as_retriever(
    search_kwargs={"k": 3}
)

print("Retriever created (k=3)")

In [ ]:
test_query = "How to craft items?"
retrieved_docs = retriever.invoke(test_query)

print(f"Query: {test_query}")
print(f"Number of chunks retrieved: {len(retrieved_docs)}")
print()
for i, doc in enumerate(retrieved_docs):
    print(f"--- Retrieved Chunk {i+1} ---")
    print(doc.page_content)
    print()